In [3]:
import time

import pandas as pd
import numpy as np

start = time.time()

column_names = ["Position X", "Position Y", "Position Z", "TAG"]

df_TAGs = pd.read_csv(r"C:\Users\Usuario\Desktop\interferências.csv",
                      header=None, names=column_names)

df_TAGs = df_TAGs.drop("Position Z", axis=1)

categories = {
    "AC_DC": "Crossing AC Cables with DC Cables",
    "AC_DRN": "Crossing AC Cables with Civil Drainage",
    "AC_SEC": "Crossing AC Cables with Secondary Interferences",
    "DC_DRN": "Crossing DC Cables with Civil Drainage",
    "MV_DRN": "Crossing MV Cables with Civil Drainage",
    "MV_AC": "Crossing MV Cables with AC Cables",
    "MV_SEC": "Crossing MV Cables with Secondary Interferences"
}

dfs = {
    name: pd.DataFrame(columns=["Position X", "Position Y", "TAG", "Layer"])
    for name in categories.keys()
}


def allocate(df_name, i):
    df = dfs[df_name]
    new_row = {"Position X": i[0], "Position Y": i[1], "TAG": i[2], "Layer": df_name}
    df.loc[len(df)] = new_row


def sort_by_xy_coordinates(df):
    return df.sort_values(by=["Position Y", "Position X"], ascending=[False, True]).reset_index(drop=True)


def tagging(df):
    return df["TAG"].astype(str).str[:-3] + (df.index + 1).astype(str).str.zfill(3)


for item in df_TAGs.values:
    if "LVAC.LVDC" in item[2]:
        allocate("AC_DC", item)
        continue
    elif "LVAC.DRN" in item[2]:
        allocate("AC_DRN", item)
        continue
    elif "LVDC.DRN" in item[2]:
        allocate("DC_DRN", item)
        continue
    elif "MV.DRN" in item[2]:
        allocate("MV_DRN", item)
        continue
    elif "MV.LVAC" in item[2]:
        allocate("MV_AC", item)
        continue
    elif "LVAC" in item[2]:
        allocate("AC_SEC", item)
        continue
    elif "MV" in item[2]:
        allocate("MV_SEC", item)

dfs["MV_SEC"]["Type of Crossing"] = dfs["MV_SEC"]["TAG"].astype(str).str[3:-4]
dfs["AC_SEC"]["Type of Crossing"] = dfs["AC_SEC"]["TAG"].astype(str).str[3:-4]

for type_of_crossing in dfs:
    print(type_of_crossing)
    dfs[type_of_crossing] = sort_by_xy_coordinates(dfs[type_of_crossing])
    dfs[type_of_crossing]["TAG"] = tagging(dfs[type_of_crossing])


dfs["MV_SEC"] = dfs["MV_SEC"].sort_values(by=["Type of Crossing", "Position Y", "Position X"], ascending=[True, False, True])
groups = (dfs["MV_SEC"]["Type of Crossing"] != dfs["MV_SEC"]["Type of Crossing"].shift()).cumsum()
dfs["MV_SEC"]["INDEX"] = dfs["MV_SEC"].groupby(groups).cumcount() + 1
dfs["MV_SEC"]["TAG"] = dfs["MV_SEC"]["TAG"].astype(str).str[:-3] + (dfs["MV_SEC"]["INDEX"]).astype(str).str.zfill(3)

dfs["AC_SEC"] = dfs["AC_SEC"].sort_values(by=["Type of Crossing", "Position Y", "Position X"], ascending=[True, False, True])
groups = (dfs["AC_SEC"]["Type of Crossing"] != dfs["AC_SEC"]["Type of Crossing"].shift()).cumsum()
dfs["AC_SEC"]["INDEX"] = dfs["AC_SEC"].groupby(groups).cumcount() + 1
dfs["AC_SEC"]["TAG"] = dfs["AC_SEC"]["TAG"].astype(str).str[:-3] + (dfs["AC_SEC"]["INDEX"]).astype(str).str.zfill(3)

print(dfs["AC_SEC"])

scr_filename = r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Tagueamento Cruzamentos\Riodades - reinsert_tags.scr"

with open(scr_filename, 'w') as scr_file:
    current_layer = []
    all_leaders = []
    for df in dfs.values():
        for _, row in df.iterrows():
            all_leaders.append((
                row["Layer"],
                row["Position X"],
                row["Position Y"],
                row["TAG"]
            ))

    # Process leaders with correct command spacing
    for layer, x, y, tag in sorted(all_leaders, key=lambda x: x[0]):
        if layer != current_layer:
            scr_file.write(f"-LAYER S Crossing-{layer}\n")
            if current_layer is not None:  # Only add newline if not first command
                scr_file.write("\n")
            current_layer = layer

        scr_file.write("MLEADER\n")
        scr_file.write(f"{x},{y}\n")
        scr_file.write(f"{x-10},{y+5}\n")
        scr_file.write(f"{tag}\n")
        # No extra newline here - critical!

    scr_file.write("\n")  # Final newline only at end

scr_filename = r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Tagueamento Cruzamentos\Riodades - create_layers.scr"

with open(scr_filename, 'w') as scr_file:
    for layer_name in categories.keys():
        scr_file.write(f"_-LAYER _M Crossing-{layer_name} \n")

AC_DC
AC_DRN
AC_SEC
DC_DRN
MV_DRN
MV_AC
MV_SEC
   Position X   Position Y                TAG   Layer Type of Crossing  INDEX
0  55338.0005  150563.6217  CR.LVAC.INTAC-001  AC_SEC       LVAC.INTAC      1
1  55313.6160  150563.4018  CR.LVAC.INTAC-002  AC_SEC       LVAC.INTAC      2
2  55590.6818  150555.1350  CR.LVAC.INTAC-003  AC_SEC       LVAC.INTAC      3
3  55585.2446  150553.4320  CR.LVAC.INTAC-004  AC_SEC       LVAC.INTAC      4
4  54854.2633  150425.5137  CR.LVAC.INTAC-005  AC_SEC       LVAC.INTAC      5
6  55258.9609  149928.1003  CR.LVAC.INTAC-006  AC_SEC       LVAC.INTAC      6
5  54864.3798  150271.6152   CR.LVAC.WALL-001  AC_SEC        LVAC.WALL      1
7  55557.3030  149743.0928   CR.LVAC.WALL-002  AC_SEC        LVAC.WALL      2
8  55587.0369  149725.8598   CR.LVAC.WALL-003  AC_SEC        LVAC.WALL      3
